# Bike Rental Demand — Project Summary and Model Comparison

A concise synthesis of the modelling work. Use this after the detailed notebooks for the high-level comparison, key validation lesson and final takeaways.

> **Portfolio note:** This notebook is a curated version of completed MSc coursework. The analytical code and saved outputs come from the original work; presentation, headings, file paths and explanatory text have been cleaned for portfolio use. The dataset itself is not redistributed here.


# Bike Rental Demand Prediction

### Portfolio version

This notebook summarizes my regression work on daily bicycle-rental demand. The original coursework brief is not reproduced here; the description and commentary below are written independently.

**Objective:** estimate future daily rental demand using calendar, weather and historical-demand information while preserving chronological order during evaluation.


## 1. Approach

The project evolved from simple regression baselines toward nonlinear models and time-aware feature engineering.

The main modelling decisions were:

1. preserve temporal order during validation;
2. create lagged and rolling features using only information available before the prediction date;
3. compare linear and nonlinear regression models;
4. tune model hyperparameters using cross-validation rather than the final evaluation period.


## 2. Feature engineering

The experiments included:

- calendar variables such as season, month, weekday, holidays and working days;
- apparent temperature, humidity, wind speed and weather situation;
- cyclical representations of month/weekday;
- demand lags such as two-day and seven-day lagged counts;
- rolling demand statistics;
- lagged/rolling temperature and humidity variables.

The lag structure was designed to avoid using the current target value as a predictor.


In [ ]:
# Representative feature-engineering pattern used in the experiments
import numpy as np
import pandas as pd

def add_time_features(df):
    df = df.copy()
    df["dteday"] = pd.to_datetime(df["dteday"])
    df = df.sort_values("dteday")

    df["year_month"] = df["dteday"].dt.to_period("M")
    df["mnth_sin"] = np.sin(2 * np.pi * df["mnth"] / 12)
    df["mnth_cos"] = np.cos(2 * np.pi * df["mnth"] / 12)

    df["cnt_lag2"] = df["cnt"].shift(2)
    df["cnt_lag7"] = df["cnt"].shift(7)
    df["cnt_roll7_lag2"] = (
        df["cnt"].shift(2).rolling(window=7, closed="left").mean()
    )

    return df


## 3. Time-aware validation

A custom expanding-window cross-validator was used in the detailed experiments. Earlier months form the training set and later months form the validation set.

This avoids an unrealistic setup in which information from later dates influences a model evaluated on earlier dates.


## 4. Models explored

The model-development notebooks cover:

- Linear Regression
- Ridge Regression
- Random Forest Regressor
- LightGBM Regressor
- Multi-Layer Perceptron (MLP)

Grid search and Optuna were used in several experiments to tune model complexity and regularization.


In [ ]:
# Results persisted in the completed experiment notebooks
import pandas as pd

saved_results = pd.DataFrame({
    "Experiment": [
        "Linear Regression",
        "Ridge + polynomial features",
        "Tuned Ridge pipeline",
        "MLP (saved test result)"
    ],
    "R2": [0.187, 0.261, 0.424, 0.528],
    "RMSE": [None, None, None, 1288.252]
})

saved_results


### Reading the results

The temporal folds showed large differences in performance. That matters more than presenting one flattering score: the model's reliability changes across time.

For example, the saved Linear Regression experiment produced fold R² values ranging from approximately **0.02 to 0.81**. This variability motivated experimentation with nonlinear models, regularization and richer lagged features.


## 5. Model interpretation

The LightGBM experiment was also used to inspect feature importance. Among the most frequently used predictive variables in the saved model were:

- apparent temperature;
- humidity;
- wind speed;
- lagged rental count;
- lagged/rolling humidity;
- weekly lagged rental count;
- weather situation and calendar features.

These results are consistent with the idea that both environmental conditions and recent demand history contribute to next-day rental volume.


## 6. What I would improve next

For a production-oriented version I would:

- formalize a single final holdout period;
- consolidate feature generation into reusable functions;
- compare models under exactly the same folds and feature set;
- add experiment tracking;
- evaluate prediction intervals, not only point predictions;
- investigate drift between the two observed years.

The detailed experimental notebooks are intentionally kept outside this portfolio summary so the main project remains readable.
